# 01 Preprocessing

Google Colab notebook version.

In [ ]:

# ============================================================
# PREPROCESSING: loading, cleaning, feature engineering, scaling, split
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def load_main_dataset(excel_path="/content/dataset"):
    df = pd.read_excel(excel_path)

    df.rename(columns={
        "Timestamp": "t",
        "Air Temperature": "temp",
        "Dew Point": "dew_point",
        "Saturation Vapor Pressure": "svp",
        "Relative Humidity": "humidity",
        "Wind Speed Avg": "wind_speed_avg",
        "Wind Speed Instant": "wind_speed_inst",
        "Wind Direction": "wind_direction",
    }, inplace=True)

    required = [
        "t", "temp", "dew_point", "svp", "humidity",
        "wind_speed_avg", "wind_speed_inst", "wind_direction", "GSR"
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Available columns: {list(df.columns)}")

    df["t"] = pd.to_datetime(df["t"], errors="coerce")
    df = df.dropna(subset=["t"]).sort_values("t").drop_duplicates(subset=["t"], keep="first")
    df.set_index("t", inplace=True)

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

def apply_quality_control(df):
    df = df.copy()

    df.loc[(df["GSR"] < 0) | (df["GSR"] > 1400), "GSR"] = np.nan
    df.loc[(df["temp"] < -40) | (df["temp"] > 60), "temp"] = np.nan
    df.loc[(df["humidity"] < 0) | (df["humidity"] > 100), "humidity"] = np.nan
    df.loc[(df["wind_speed_avg"] < 0) | (df["wind_speed_avg"] > 75), "wind_speed_avg"] = np.nan
    df.loc[(df["wind_speed_inst"] < 0) | (df["wind_speed_inst"] > 75), "wind_speed_inst"] = np.nan
    df.loc[(df["wind_direction"] < 0) | (df["wind_direction"] > 360), "wind_direction"] = np.nan

    return df

def add_engineered_features(df):
    df = df.copy()

    # Interpolate missing values
    df = df.interpolate(method="time").ffill().bfill()

    # Hour-of-day cyclical encoding
    df["hour"] = df.index.hour
    df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24)
    df.drop(columns=["hour"], inplace=True)

    # Day-of-year cyclical encoding
    df["doy"] = df.index.dayofyear
    df["sin_doy"] = np.sin(2 * np.pi * df["doy"] / 365.25)
    df["cos_doy"] = np.cos(2 * np.pi * df["doy"] / 365.25)
    df.drop(columns=["doy"], inplace=True)

    # Wind direction cyclical encoding
    df["sin_wind_dir"] = np.sin(2 * np.pi * df["wind_direction"] / 360)
    df["cos_wind_dir"] = np.cos(2 * np.pi * df["wind_direction"] / 360)
    df.drop(columns=["wind_direction"], inplace=True)

    # Rolling and lag features
    df["GSR_lag1"] = df["GSR"].shift(1)
    df["GSR_roll3"] = df["GSR"].rolling(3, min_periods=3).mean()
    df["temp_roll3"] = df["temp"].rolling(3, min_periods=3).mean()
    df["humidity_roll3"] = df["humidity"].rolling(3, min_periods=3).mean()

    df = df.dropna().copy()

    return df

def get_main_feature_list():
    METEO = ["temp", "dew_point", "svp", "humidity", "wind_speed_avg", "wind_speed_inst"]
    ENG = ["sin_wind_dir", "cos_wind_dir", "sin_hour", "cos_hour", "sin_doy", "cos_doy"]
    ROLL = ["GSR_lag1", "GSR_roll3", "temp_roll3", "humidity_roll3"]
    TARGET = "GSR"
    FEATURES = METEO + ENG + ROLL + [TARGET]
    return FEATURES, TARGET

def make_sequences_from_dataframe(
    df,
    features,
    target,
    window=48,
    train_ratio=0.70,
    val_ratio=0.15,
    remove_zero_train_val=True,
):
    n_total = len(df)

    i1_raw = int(train_ratio * n_total)
    i2_raw = int((train_ratio + val_ratio) * n_total)

    df_train = df.iloc[:i1_raw].copy()
    df_val = df.iloc[i1_raw:i2_raw].copy()
    df_test = df.iloc[i2_raw:].copy()

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    # Fit scalers on training data only
    scaler_X.fit(df_train[features])
    scaler_y.fit(df_train[[target]])

    data_X = scaler_X.transform(df[features])
    data_y = scaler_y.transform(df[[target]])

    X, y, time_index = [], [], []

    for i in range(window, len(df)):
        X.append(data_X[i-window:i])
        y.append(data_y[i, 0])
        time_index.append(df.index[i])

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    time_index = np.asarray(time_index)

    train_end_time = df_train.index[-1]
    val_end_time = df_val.index[-1]

    train_mask = time_index <= train_end_time
    val_mask = (time_index > train_end_time) & (time_index <= val_end_time)
    test_mask = time_index > val_end_time

    if remove_zero_train_val:
        target_values = df.loc[time_index, target].values
        daylight_mask = target_values > 0
        train_mask = train_mask & daylight_mask
        val_mask = val_mask & daylight_mask

    data = {
        "X_train": X[train_mask],
        "y_train": y[train_mask],
        "X_val": X[val_mask],
        "y_val": y[val_mask],
        "X_test": X[test_mask],
        "y_test": y[test_mask],
        "test_times": time_index[test_mask],
        "scaler_X": scaler_X,
        "scaler_y": scaler_y,
    }

    return data

def prepare_main_data(
    excel_path="/content/dataset",
    window=48,
    train_ratio=0.70,
    val_ratio=0.15,
    remove_zero_train_val=True,
):
    df = load_main_dataset(excel_path)
    df = apply_quality_control(df)
    df = add_engineered_features(df)
    FEATURES, TARGET = get_main_feature_list()

    data = make_sequences_from_dataframe(
        df=df,
        features=FEATURES,
        target=TARGET,
        window=window,
        train_ratio=train_ratio,
        val_ratio=val_ratio,
        remove_zero_train_val=remove_zero_train_val,
    )

    return df, FEATURES, TARGET, data


In [ ]:
# Example use
set_seed(42)

df, FEATURES, TARGET, data = prepare_main_data(
    excel_path="dataset",
    window=48,
    train_ratio=0.70,
    val_ratio=0.15,
    remove_zero_train_val=True,
)

print("Cleaned records:", len(df))
print("Features:", FEATURES)
print("X_train:", data["X_train"].shape)
print("X_val:", data["X_val"].shape)
print("X_test:", data["X_test"].shape)